# Generating Names with an RNN

Last week the RNN **read** a name and predicted its language (many-to-one).
Today it **writes** names: we train a character-level **language model** that predicts the next character of a name.

The same four steps as on the slides:

1. **Training data**: the target is the input shifted by one character.
2. **Loss**: one prediction and one loss term **per time step**, summed up.
3. **Training**: backpropagation through time on the summed loss.
4. **Generation**: predict a character, sample it, feed it back in, repeat until the end.

The `assert` cells check your solution. If an assert fails, read the message and fix your code before you continue.

## Preparing the data

We use the names data from last week, but only **one language** at a time (default: German, about 700 names).
All names are converted to lower case, so the model has fewer characters to learn.
Two special tokens mark the start and the end of a name:

* `^` (**start token**): the first input, before the model has seen any letter
* `$` (**end token**): the model predicts it when the name is complete

In [ ]:
import string, time, random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
random.seed(0)

language = "German"   # try later: "Italian", "Japanese", "Russian" (bigger, slower), ...
names = sorted(set(open(f'data/names/{language}.txt').read().lower().split()))

SOS, EOS = '^', '$'
all_chars = string.ascii_lowercase + SOS + EOS   # our vocabulary: a-z, start and end token
n_chars = len(all_chars)

print(len(names), language, "names,", n_chars, "characters in the vocabulary")
print("Examples:", random.sample(names, 8))

Two helper functions (you know the first one from last week):

* `nameToTensor(s)`: one-hot encoding, shape `<len(s) x 1 x n_chars>` (input of the RNN)
* `nameToIndices(s)`: the index of every character, shape `<len(s)>` (target for `CrossEntropyLoss`)

In [ ]:
def nameToTensor(s):
    tensor = torch.zeros(len(s), 1, n_chars)
    for t, char in enumerate(s):
        tensor[t][0][all_chars.find(char)] = 1
    return tensor

def nameToIndices(s):
    return torch.tensor([all_chars.find(char) for char in s])

print(nameToTensor("^abe").shape, nameToIndices("abe$"))

---
## Step 1: Training pairs

The RNN reads the name character by character and has to predict the **next** character at every step:

```
input  x:   ^   w   e   b   e   r
target y:   w   e   b   e   r   $
```

The target is the input shifted by one position. No labels are needed: the name is its own label.

### Task 1

Write `makeTrainingPair(name)`. It returns the input tensor (one-hot, starting with `^`) and the target tensor (indices, ending with `$`).
Use the two helper functions.

In [ ]:
def makeTrainingPair(name):
    # TODO
    pass

In [ ]:
pair = makeTrainingPair("weber")
assert pair is not None, "makeTrainingPair returns nothing yet."
x, y = pair
assert x.shape == (6, 1, n_chars), f"Input must have shape <6 x 1 x {n_chars}> for 'weber', got {tuple(x.shape)}"
assert y.shape == (6,), f"Target must have shape <6> for 'weber', got {tuple(y.shape)}"
assert x[0, 0, all_chars.find(SOS)] == 1, "The input must start with the start token ^"
assert y[-1] == all_chars.find(EOS), "The target must end with the end token $"
assert x[1, 0].argmax() == y[0], "Input at step t+1 must be the target of step t (shift by one)"
print("OK")

---
## Step 2: The model and the loss

We use **the RNN from last week**, unchanged. The only difference: the output now has one score per **character**
(`n_chars`) instead of one score per language.

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.input2hidden = nn.Linear(input_size, hidden_size)    # W_xh
        self.hidden2hidden = nn.Linear(hidden_size, hidden_size)  # W_hh
        self.hidden2output = nn.Linear(hidden_size, output_size)  # W_hy

    def forward(self, x, hidden):
        hidden = torch.tanh(self.input2hidden(x) + self.hidden2hidden(hidden))
        output = self.hidden2output(hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)

torch.manual_seed(0)
rnn = RNN(n_chars, 128, n_chars)
criterion = nn.CrossEntropyLoss()

### Task 2: Loss of one name

Last week only the **last** output was used (many-to-one). Now the RNN makes a prediction at **every** time step (many-to-many):

$$L = L_1 + L_2 + \dots + L_T$$

Write `nameLoss(rnn, name)`, which returns the loss for a given name. This loss is later used to do backpropagation on.

The function should do:
1. Create the training pair and the initial hidden state.
2. Feed the input step by step into the RNN and carry the hidden state.
3. At every step, compute the loss between the output and the target character, and add it up.

Hint: `criterion` expects a target of shape `<1>`, so use `target_tensor[t:t+1]` instead of `target_tensor[t]`.

In [ ]:
def nameLoss(rnn, name):
    # TODO
    pass

In [ ]:
import math
loss = nameLoss(rnn, "abe")
assert loss is not None, "nameLoss returns nothing yet."
assert loss.requires_grad, "The loss must be computed with the RNN, so that we can call backward() on it."
assert 3.5 * math.log(n_chars) < loss.item() < 4.5 * math.log(n_chars), \
    f"For 'abe' we expect 4 loss terms of about ln({n_chars}) = {math.log(n_chars):.2f} each (untrained model), got {loss.item():.2f}. Did you sum over all steps?"
print(f"OK: loss {loss.item():.2f}, i.e. {loss.item() / 4:.2f} per character")

**Question:** Before training, the loss per character is close to $\ln(28) \approx 3.33$. Why exactly this value?

*Your answer:* ...

---
## Step 3: Training

### Task 3

Complete the training loop. For every name: reset the gradients, compute the loss with `nameLoss`,
backpropagate (backpropagation through time) and take an optimizer step.

We print the mean loss **per character** (the summed loss divided by the number of predicted characters), so that long and short names count equally.
20 epochs take about 30 seconds to 2 minutes, depending on your laptop.

In [ ]:
optimizer = torch.optim.Adam(rnn.parameters(), lr=0.003)
losses = []

for epoch in range(1, 21):
    start = time.time()
    total_loss, total_chars = 0, 0
    for i in torch.randperm(len(names)).tolist():
        # TODO: 4 lines: reset gradients, compute the loss of names[i], backpropagate, optimizer step

        total_loss += loss.item()
        total_chars += len(names[i]) + 1
    losses.append(total_loss / total_chars)
    print(f"Epoch {epoch}: loss per character {losses[-1]:.3f} ({time.time() - start:.0f} s)")

plt.plot(range(1, len(losses) + 1), losses, marker='o')
plt.xlabel("epoch"); plt.ylabel("loss per character"); plt.show()

In [ ]:
assert len(losses) == 20, "Train for 20 epochs."
assert losses[-1] < losses[0] - 0.3, "The loss should drop clearly. Did you call optimizer.step()?"
assert losses[-1] < 1.85, f"Expected a final loss below 1.85, got {losses[-1]:.3f}"
print("OK")

---
## Step 4: Generating names

The trained RNN outputs scores for the next character. A softmax turns them into probabilities, e.g. after `^schm`:

```
i: 0.55   a: 0.20   e: 0.15   ...
```

There are two ways to pick the next character (see slides):

* **Greedy**: always take the most likely character (`argmax`).
* **Sampling**: draw a character according to the probabilities (`torch.multinomial`).

### Task 4

Write `sampleNext(output, greedy=False)`. It gets the output of the RNN (shape `<1 x n_chars>`) and returns the chosen **character** as a string.

In [ ]:
def sampleNext(output, greedy=False):
    # TODO
    pass

In [ ]:
# a fake output: 'm' with probability 0.6, 'f' with 0.3, 'r' with 0.1
fake = torch.full((1, n_chars), -1e9)
for char, p in [('m', 0.6), ('f', 0.3), ('r', 0.1)]:
    fake[0, all_chars.find(char)] = math.log(p)

assert sampleNext(fake, greedy=True) == 'm', "Greedy must return the most likely character as a string"
samples = [sampleNext(fake) for _ in range(2000)]
assert set(samples) <= {'m', 'f', 'r'}, "Sampling must only return characters with probability > 0"
share = {c: samples.count(c) / len(samples) for c in 'mfr'}
assert abs(share['m'] - 0.6) < 0.05 and abs(share['f'] - 0.3) < 0.05, \
    f"Sampling must follow the probabilities, got {share}"
print("OK:", {c: round(v, 2) for c, v in share.items()})

### Task 5

Write `generate(rnn, prompt="", greedy=False, max_length=15)`:

1. Feed the start token `^` and then the characters of the `prompt` into the RNN (ignore the outputs, but keep the hidden state).
2. Choose the next character with `sampleNext`.
3. If it is the end token `$`, stop. Otherwise append it to the name and **feed it back in as the next input**.
4. Repeat until `$` or `max_length` characters.

Return the name (including the prompt) with the first letter in upper case (`name.capitalize()`).
We do not train here, so wrap everything in `torch.no_grad()`.

In [ ]:
def generate(rnn, prompt="", greedy=False, max_length=15):
    with torch.no_grad():
        # TODO
        pass

In [ ]:
torch.manual_seed(1)
generated = [generate(rnn) for _ in range(20)]
assert all(isinstance(g, str) for g in generated), "generate must return a string"
assert all(len(g) <= 15 for g in generated), "Names must not be longer than max_length"
assert all(set(g.lower()) <= set(string.ascii_lowercase) for g in generated), "Names must not contain ^ or $"
assert len(set(generated)) > 10, "Sampling should give different names in different runs"
assert generate(rnn, "schm").startswith("Schm"), "The name must start with the prompt"
assert generate(rnn, "k", greedy=True) == generate(rnn, "k", greedy=True), "Greedy decoding must be deterministic"
print("OK:", generated)

---
## Step 5: Experiments

### Task 6

1. Generate names with **greedy decoding** for the prompts `""`, `"a"`, `"b"`, `"k"`, `"m"`, `"s"`, and compare them with **sampled** names for the same prompts.
   What do you observe? Which method would you use to suggest new names?
2. Has the model only memorized the training data? Generate 100 names with sampling and compute the share of names that are **not** in the training data
   (careful: `names` is in lower case). Store it in `novel_share`.

In [ ]:
for prompt in ["", "a", "b", "k", "m", "s"]:
    # TODO: print the greedy name and three sampled names for this prompt
    pass

novel_share = None  # TODO

In [ ]:
assert novel_share is not None, "Compute novel_share first."
assert 0 <= novel_share <= 1, "novel_share is a share between 0 and 1"
print(f"OK: {novel_share:.0%} new names")

*Your answers:* ...

**Questions** (answer in a few sentences):

1. Sometimes the model generates a very short name or even an empty one. Explain this with the generation loop.
2. Why do we need the end token `$`? What would happen without it?
3. The greedy name for the empty prompt is always the same. Why?

*Your answers:* ...

### Bonus: Temperature and other languages

* **Temperature**: divide the output by a temperature $T$ before the softmax (`torch.softmax(output / T, dim=1)`). Try $T = 0.5$ and $T = 1.5$.
  How do the names change? What happens for very small $T$?
* Change `language` at the top (e.g. `"Italian"` or `"Japanese"`), run the notebook again and look at the generated names.